In [ ]:
# @title 1. Installing Dependencies
import os
import sys

# FTF
print("Installing heavy libraries...")
!pip install -q markdown xhtml2pdf
!pip install paddlepaddle-gpu==2.6.1
!pip install paddleocr==2.7.3
!apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx
!pip install accelerate bitsandbytes
!pip install fastapi uvicorn python-multipart pyngrok nest-asyncio
!pip install "langchain==0.1.20" "langchain-community==0.0.38"
!pip install "transformers==4.41.2" --force-reinstall


print("Locking versions together...")
!pip install "numpy==1.26.4" "opencv-python-headless==4.8.0.74" "opencv-contrib-python-headless==4.8.0.74" --force-reinstall

print("\n INSTALLATION COMPLETE.")
print("restart now")

In [ ]:
# @title 2. Logic Engine
import torch
from paddleocr import PaddleOCR
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from PIL import Image, ImageEnhance
import numpy as np
import io
import warnings
import sys
import subprocess
import gc

# Memory Cleanup
if 'digitizer' in locals(): del digitizer
gc.collect()
torch.cuda.empty_cache()

warnings.filterwarnings("ignore")

# GPU check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Uing device: {device}")

class NotesDigitizer:
    def __init__(self):
        print("Loading OCR...")
        self.ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

        print("Loading LLM ...")
        model_id = "microsoft/Phi-3-mini-4k-instruct"
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            trust_remote_code=True,
            device_map="auto",
            attn_implementation="eager"
        )

        self.llm_pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=2500,
            do_sample=True,
            temperature=0.1
        )
        print("Systems Online.")

    def preprocess_image(self, image_bytes):
        img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(1.8)
        enhancer = ImageEnhance.Sharpness(img)
        img = enhancer.enhance(1.5)
        return np.array(img)

    def process_batch(self, image_files_bytes):
        """Process multiple images, extract text with OCR and structure with LLM."""
        clean_pages = []
        total_pages = len(image_files_bytes)
        print(f"Processing {total_pages} pages individually...")

        # OCR Phase
        for idx, img_bytes in enumerate(image_files_bytes):
            try:
                enhanced_image = self.preprocess_image(img_bytes)
                result = self.ocr_engine.ocr(enhanced_image, cls=True)
                page_text_lines = []
                if result and result[0]:
                    for line in result[0]:
                        text = line[1][0]
                        if "_____" in text:
                            continue
                        page_text_lines.append(text)

                # Format this page's text
                raw_text = "\n".join(page_text_lines)
                clean_pages.append(f"--- PAGE {idx+1} ---\n{raw_text}")
                print(f" Page {idx+1} processed successfully")

            except Exception as e:
                print(f" Error on Page {idx+1}: {str(e)[:100]}...")
                clean_pages.append(f"--- PAGE {idx+1} (FAILED) ---\n[OCR Error: {str(e)[:50]}...]")

        # Combine all pages for LLM processing
        full_context = "\n\n".join(clean_pages)

        # LLM Phase
        print("Structuring notes with LLM...")

        prompt = f"""<|user|>
You are a strict University Professor. You have been given a student's unordered, chaotic lecture notes.
Your task is to RESTRUCTURE them into a perfect study guide.

Here is the chaotic raw text from multiple pages:
{full_context}

RULES FOR RESTRUCTURING:
1. **Ignore Page Order**: Group facts by TOPIC, not by page number. If Page 1 and Page 8 discuss the same concept, merge them.
2. **Create Logical Headers**: Use # Main Topic and ## Subtopic.
3. **Fix Continuity**: If a sentence cuts off on one page and resumes on another, merge it.
4. **Format**: Use Markdown. Use bullet points for lists. Use code blocks for code.

Output the final study guide now.
<|end|>
<|assistant|>"""

        try:
            output = self.llm_pipe(
                prompt,
                max_new_tokens=2500,
                temperature=0.3,
                do_sample=True
            )

            final_notes = output[0]['generated_text']


            if "<|assistant|>" in final_notes:
                final_notes = final_notes.split("<|assistant|>")[1].strip()

            print("Notes structured successfully!")
            return final_notes

        except Exception as e:
            print(f"LLM error: {e}")
            # Fallback
            return f"""# Recovery Mode - Raw OCR Text

The LLM failed to structure the notes. Here is the raw extracted text:

{full_context}

*Note: This is unprocessed OCR output. It may contain errors and lacks structure.*"""

# Instantiating
digitizer = NotesDigitizer()

In [ ]:
# @title 3. Dashboard
from fastapi import FastAPI, File, UploadFile, Request, Response
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import os
import io
import markdown
from xhtml2pdf import pisa
from typing import List

app = FastAPI()

# temp dir
os.makedirs("templates", exist_ok=True)
templates = Jinja2Templates(directory="templates")

# pdf gen helper
def convert_markdown_to_pdf(markdown_text):
    html_body = markdown.markdown(markdown_text)
    full_html = f"""
    <html>
    <head>
        <style>
            body {{ font-family: Helvetica, sans-serif; font-size: 12pt; color: #333; }}
            h1 {{ color: #4f46e5; border-bottom: 2px solid #ddd; padding-bottom: 5px; }}
            h2 {{ color: #334155; margin-top: 20px; }}
            code {{ background-color: #f3f4f6; padding: 2px 5px; font-family: Courier; }}
            pre {{ background-color: #f1f5f9; padding: 10px; border-radius: 5px; }}
            ul {{ line-height: 1.6; }}
        </style>
    </head>
    <body>
        <div style="text-align: center; margin-bottom: 30px;">
            <h1 style="border:none;">Lecturify Notes</h1>
            <p style="color: #666; font-size: 10pt;">Generated by AI</p>
        </div>
        {html_body}
    </body>
    </html>
    """

    pdf_buffer = io.BytesIO()
    pisa_status = pisa.CreatePDF(full_html, dest=pdf_buffer)

    if pisa_status.err:
        return None
    return pdf_buffer.getvalue()



# api endpoints
class PDFRequest(BaseModel):
    notes: str

@app.get("/", response_class=HTMLResponse)
async def home(request: Request):
    return templates.TemplateResponse("index.html", {"request": request})

@app.post("/digitize-batch")
async def digitize_batch(files: List[UploadFile] = File(...)):
    file_bytes_list = []
    for file in files:
        content = await file.read()
        file_bytes_list.append(content)

    result_markdown = digitizer.process_batch(file_bytes_list)
    return {"notes": result_markdown}

@app.post("/generate-pdf")
async def generate_pdf(req: PDFRequest):
    pdf_bytes = convert_markdown_to_pdf(req.notes)
    if not pdf_bytes:
        return {"error": "PDF generation failed"}

    return Response(content=pdf_bytes, media_type="application/pdf")

print("Server ready.")

In [ ]:
# @title 4. Going live
import threading
from pyngrok import ngrok
import uvicorn
import nest_asyncio


NGROK_AUTH_TOKEN = "....... get one for yourself"

ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

try:
    public_url = ngrok.connect(8000).public_url
    print(f"\n Public URL: {public_url}")
except Exception as e:
    print(f"Error starting tunnel: {e}")

def run_server():
    nest_asyncio.apply()
    uvicorn.run(app, port=8000, host="127.0.0.1", log_level="info")

thread = threading.Thread(target=run_server)
thread.start()